# Cell 1 — Imports

In [1]:
import os, glob, zipfile, json, shutil, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Cell 2 — Output folder

In [2]:
OUT_DIR = Path("/kaggle/working/output/phase11b")
OUT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR = OUT_DIR / "extracted"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_DIR:", OUT_DIR)

OUT_DIR: /kaggle/working/output/phase11b


# Cell 3 — Find all Phase-11 zip files

In [3]:
from pathlib import Path
import zipfile

SEARCH_ROOT = Path("/kaggle/input")

# Find the Phase-11 root anywhere inside Kaggle input
phase11_roots = [p for p in SEARCH_ROOT.rglob("Phase-11") if p.is_dir()]
assert len(phase11_roots) > 0, "Could not find a Phase-11 folder under /kaggle/input"

PHASE11_ROOT = phase11_roots[0]
print("Using PHASE11_ROOT:", PHASE11_ROOT)

# Collect sources:
#   1) zip files under Phase-11
#   2) already-extracted run folders that contain *_val_summary_report.csv
zip_sources = [
    p for p in PHASE11_ROOT.rglob("*.zip")
    if "phase11" in p.name.lower() or "cracksafe" in p.name.lower()
]

dir_sources = sorted({p.parent for p in PHASE11_ROOT.rglob("*_val_summary_report.csv")})

phase11_sources = sorted(set(zip_sources + dir_sources), key=lambda x: str(x))

print("Found Phase-11 sources:", len(phase11_sources))
for p in phase11_sources:
    kind = "ZIP" if p.is_file() else "DIR"
    print(f" - [{kind}] {p}")

assert len(phase11_sources) >= 2, "Need at least 2 Phase-11 run sources."

Using PHASE11_ROOT: /kaggle/input/datasets/abarman79/phase11b-ablation-aggregation-finalist-selection00/Phase-11
Found Phase-11 sources: 5
 - [DIR] /kaggle/input/datasets/abarman79/phase11b-ablation-aggregation-finalist-selection00/Phase-11/phase11-cracksafe-mrc-training_Ablation_All_True
 - [DIR] /kaggle/input/datasets/abarman79/phase11b-ablation-aggregation-finalist-selection00/Phase-11/phase11-cracksafe-mrc-training_ablationAll_ThreeTrue
 - [DIR] /kaggle/input/datasets/abarman79/phase11b-ablation-aggregation-finalist-selection00/Phase-11/phase11-cracksafe-mrc-training_ablationAll_TwoTrue
 - [DIR] /kaggle/input/datasets/abarman79/phase11b-ablation-aggregation-finalist-selection00/Phase-11/phase11-cracksafe-mrc-training_ablationAll_false
 - [DIR] /kaggle/input/datasets/abarman79/phase11b-ablation-aggregation-finalist-selection00/Phase-11/phase11-cracksafe-mrc-training_ablationOne_True


# Cell 4 — Extract zip files

In [4]:
run_dirs = []

for src in phase11_sources:
    src = Path(src)

    # Case A: already-extracted run directory
    if src.is_dir():
        summary_files = list(src.glob("*_val_summary_report.csv"))

        # if the CSV is deeper inside, search recursively
        if len(summary_files) == 0:
            summary_files = list(src.rglob("*_val_summary_report.csv"))

        if len(summary_files) == 0:
            print("No valid output folder found in directory:", src)
            continue

        run_dirs.append(summary_files[0].parent)
        continue

    # Case B: zip file
    if src.is_file() and zipfile.is_zipfile(src):
        run_slug = src.stem
        target = EXTRACT_DIR / run_slug
        target.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(src, "r") as z:
            z.extractall(target)

        summary_files = list(target.rglob("*_val_summary_report.csv"))
        if len(summary_files) == 0:
            print("No valid output folder found in zip:", src)
            continue

        run_dirs.append(summary_files[0].parent)
        continue

    print("Skipping unsupported source:", src)

# remove duplicates while preserving order
seen = set()
run_dirs = [d for d in run_dirs if not (str(d) in seen or seen.add(str(d)))]

print("Usable Phase-11 run directories:", len(run_dirs))
for d in run_dirs:
    print(" -", d)

assert len(run_dirs) >= 2, "No usable Phase-11 run directories found."

Usable Phase-11 run directories: 5
 - /kaggle/input/datasets/abarman79/phase11b-ablation-aggregation-finalist-selection00/Phase-11/phase11-cracksafe-mrc-training_Ablation_All_True
 - /kaggle/input/datasets/abarman79/phase11b-ablation-aggregation-finalist-selection00/Phase-11/phase11-cracksafe-mrc-training_ablationAll_ThreeTrue
 - /kaggle/input/datasets/abarman79/phase11b-ablation-aggregation-finalist-selection00/Phase-11/phase11-cracksafe-mrc-training_ablationAll_TwoTrue
 - /kaggle/input/datasets/abarman79/phase11b-ablation-aggregation-finalist-selection00/Phase-11/phase11-cracksafe-mrc-training_ablationAll_false
 - /kaggle/input/datasets/abarman79/phase11b-ablation-aggregation-finalist-selection00/Phase-11/phase11-cracksafe-mrc-training_ablationOne_True


# Cell 5 — Read summary/config/history from every run

In [5]:
records = []

for run_dir in run_dirs:
    summary_path = list(run_dir.glob("*_val_summary_report.csv"))[0]
    history_path = list(run_dir.glob("*_history.csv"))[0]
    config_path = list(run_dir.glob("*_train_config.json"))[0]

    summary = pd.read_csv(summary_path).iloc[0].to_dict()
    history = pd.read_csv(history_path)
    config = json.loads(Path(config_path).read_text(encoding="utf-8"))

    record = {
        "folder_name": run_dir.name,
        "run_name": config.get("run_name", run_dir.name),

        "use_boundary": config.get("use_boundary", False),
        "use_cldice": config.get("use_cldice", False),
        "use_lovasz": config.get("use_lovasz", False),
        "use_fda": config.get("use_fda", False),
        "use_mrc": config.get("use_mrc", False),

        "mean_iou_all": summary.get("mean_iou_all", np.nan),
        "mean_iou_crack_only": summary.get("mean_iou_crack_only", np.nan),
        "mean_dice_crack_only": summary.get("mean_dice_crack_only", np.nan),
        "no_crack_fp_rate": summary.get("no_crack_fp_rate", np.nan),
        "best_epoch": summary.get("best_epoch", np.nan),

        "last_train_loss": history.iloc[-1]["train_loss"] if len(history) else np.nan,
        "last_val_loss": history.iloc[-1]["val_loss"] if len(history) else np.nan,
        "max_val_iou_crack_only_curve": history["val_mean_iou_crack_only"].max() if "val_mean_iou_crack_only" in history.columns else np.nan,
        "min_val_fp_curve": history["val_no_crack_fp_rate"].min() if "val_no_crack_fp_rate" in history.columns else np.nan,
    }
    records.append(record)

ablation_df = pd.DataFrame(records)
ablation_df

,folder_name,run_name,use_boundary,use_cldice,use_lovasz,use_fda,use_mrc,mean_iou_all,mean_iou_crack_only,mean_dice_crack_only,no_crack_fp_rate,best_epoch,last_train_loss,last_val_loss,max_val_iou_crack_only_curve,min_val_fp_curve
0,phase11-cracksafe-mrc-training_Ablation_All_True,phase11_full_cracksafe_mrc_unetpp_effb3_448,True,True,True,True,True,0.623703,0.575611,0.717199,0.050459,12,0.658606,0.806584,0.575611,0.036697
1,phase11-cracksafe-mrc-training_ablationAll_Thr...,phase11_full_cracksafe_mrc_unetpp_effb3_448,True,True,True,False,False,0.629538,0.581629,0.722451,0.045872,12,0.628581,0.788446,0.581629,0.036697
2,phase11-cracksafe-mrc-training_ablationAll_Two...,phase11_full_cracksafe_mrc_unetpp_effb3_448,True,True,False,False,False,0.624130,0.578809,0.720286,0.068807,7,0.436651,0.513429,0.578809,0.059633
3,phase11-cracksafe-mrc-training_ablationAll_false,phase11_full_cracksafe_mrc_unetpp_effb3_448,False,False,False,False,False,0.624709,0.581505,0.721807,0.082569,12,0.369054,0.442194,0.581505,0.077982
4,phase11-cracksafe-mrc-training_ablationOne_True,phase11_full_cracksafe_mrc_unetpp_effb3_448,True,False,False,False,False,0.622460,0.582308,0.722701,0.105505,10,0.370001,0.441923,0.582308,0.073394


# Cell 6 — Create readable ablation label

In [6]:
def label_from_row(r):
    parts = []
    if r["use_boundary"]: parts.append("Boundary")
    if r["use_cldice"]:  parts.append("clDice")
    if r["use_lovasz"]:  parts.append("Lovasz")
    if r["use_fda"]:     parts.append("FDA")
    if r["use_mrc"]:     parts.append("MRC")
    if len(parts) == 0:
        return "Baseline"
    return " + ".join(parts)

ablation_df["ablation_label"] = ablation_df.apply(label_from_row, axis=1)

# remove duplicates if same label appears more than once; keep best crack IoU
ablation_df = (
    ablation_df.sort_values(["ablation_label", "mean_iou_crack_only"], ascending=[True, False])
    .drop_duplicates(subset=["ablation_label"], keep="first")
    .reset_index(drop=True)
)

ablation_df

,folder_name,run_name,use_boundary,use_cldice,use_lovasz,use_fda,use_mrc,mean_iou_all,mean_iou_crack_only,mean_dice_crack_only,no_crack_fp_rate,best_epoch,last_train_loss,last_val_loss,max_val_iou_crack_only_curve,min_val_fp_curve,ablation_label
0,phase11-cracksafe-mrc-training_ablationAll_false,phase11_full_cracksafe_mrc_unetpp_effb3_448,False,False,False,False,False,0.624709,0.581505,0.721807,0.082569,12,0.369054,0.442194,0.581505,0.077982,Baseline
1,phase11-cracksafe-mrc-training_ablationOne_True,phase11_full_cracksafe_mrc_unetpp_effb3_448,True,False,False,False,False,0.622460,0.582308,0.722701,0.105505,10,0.370001,0.441923,0.582308,0.073394,Boundary
2,phase11-cracksafe-mrc-training_ablationAll_Two...,phase11_full_cracksafe_mrc_unetpp_effb3_448,True,True,False,False,False,0.624130,0.578809,0.720286,0.068807,7,0.436651,0.513429,0.578809,0.059633,Boundary + clDice
3,phase11-cracksafe-mrc-training_ablationAll_Thr...,phase11_full_cracksafe_mrc_unetpp_effb3_448,True,True,True,False,False,0.629538,0.581629,0.722451,0.045872,12,0.628581,0.788446,0.581629,0.036697,Boundary + clDice + Lovasz
4,phase11-cracksafe-mrc-training_Ablation_All_True,phase11_full_cracksafe_mrc_unetpp_effb3_448,True,True,True,True,True,0.623703,0.575611,0.717199,0.050459,12,0.658606,0.806584,0.575611,0.036697,Boundary + clDice + Lovasz + FDA + MRC


# Cell 7 — Save paper-ready ablation table

In [7]:
table_cols = [
    "ablation_label",
    "mean_iou_all",
    "mean_iou_crack_only",
    "mean_dice_crack_only",
    "no_crack_fp_rate",
    "best_epoch",
]

table_df = ablation_df[table_cols].copy()
table_df = table_df.sort_values("mean_iou_crack_only", ascending=False).reset_index(drop=True)

csv_path = OUT_DIR / "phase11b_ablation_table.csv"
tex_path = OUT_DIR / "phase11b_ablation_table.tex"

table_df.to_csv(csv_path, index=False)
table_df.to_latex(tex_path, index=False, float_format="%.4f")

print("Saved:", csv_path)
print("Saved:", tex_path)
table_df

Saved: /kaggle/working/output/phase11b/phase11b_ablation_table.csv
Saved: /kaggle/working/output/phase11b/phase11b_ablation_table.tex


,ablation_label,mean_iou_all,mean_iou_crack_only,mean_dice_crack_only,no_crack_fp_rate,best_epoch
0,Boundary,0.622460,0.582308,0.722701,0.105505,10
1,Boundary + clDice + Lovasz,0.629538,0.581629,0.722451,0.045872,12
2,Baseline,0.624709,0.581505,0.721807,0.082569,12
3,Boundary + clDice,0.624130,0.578809,0.720286,0.068807,7
4,Boundary + clDice + Lovasz + FDA + MRC,0.623703,0.575611,0.717199,0.050459,12


# Cell 8 — Tradeoff plot

In [8]:
plt.figure(figsize=(8,6))

for _, row in ablation_df.iterrows():
    x = row["no_crack_fp_rate"]
    y = row["mean_iou_crack_only"]
    plt.scatter(x, y, s=90)
    plt.text(x, y, row["ablation_label"], fontsize=9)

plt.xlabel("No-crack FP rate")
plt.ylabel("Mean IoU (crack-only)")
plt.title("Phase-11B Ablation Tradeoff")
plt.tight_layout()

plot_path = OUT_DIR / "phase11b_tradeoff_plot.png"
plt.savefig(plot_path, dpi=220)
plt.close()

print("Saved:", plot_path)

Saved: /kaggle/working/output/phase11b/phase11b_tradeoff_plot.png


# Cell 9 — Finalist selection

In [9]:
best_iou = ablation_df["mean_iou_crack_only"].max()
pool = ablation_df[ablation_df["mean_iou_crack_only"] >= 0.98 * best_iou].copy()
finalist = pool.sort_values(["no_crack_fp_rate", "mean_iou_crack_only"], ascending=[True, False]).iloc[0]

finalist_info = {
    "selected_ablation": finalist["ablation_label"],
    "mean_iou_crack_only": float(finalist["mean_iou_crack_only"]),
    "mean_dice_crack_only": float(finalist["mean_dice_crack_only"]),
    "no_crack_fp_rate": float(finalist["no_crack_fp_rate"]),
    "rule": "lowest no-crack FP rate among ablations with crack-IoU >= 98% of best crack-IoU"
}

json_path = OUT_DIR / "phase11b_finalist_selection.json"
Path(json_path).write_text(json.dumps(finalist_info, indent=2), encoding="utf-8")

print(json.dumps(finalist_info, indent=2))
print("Saved:", json_path)

{
  "selected_ablation": "Boundary + clDice + Lovasz",
  "mean_iou_crack_only": 0.581629243609008,
  "mean_dice_crack_only": 0.7224508549884767,
  "no_crack_fp_rate": 0.0458715596330275,
  "rule": "lowest no-crack FP rate among ablations with crack-IoU >= 98% of best crack-IoU"
}
Saved: /kaggle/working/output/phase11b/phase11b_finalist_selection.json


# Cell 10 — Export table as PNG

In [10]:
fig, ax = plt.subplots(figsize=(12, 0.8 + 0.5*len(table_df)))
ax.axis("off")

tbl = ax.table(
    
    cellText=table_df.round(4).values, 
    colLabels=table_df.columns,
    loc="center"
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.4)

png_path = OUT_DIR / "phase11b_ablation_table.png"
plt.tight_layout()
plt.savefig(png_path, dpi=220, bbox_inches="tight")
plt.close()

print("Saved:", png_path)

Saved: /kaggle/working/output/phase11b/phase11b_ablation_table.png
